# Libro vs guion: la adaptación como transporte óptimo
**Proyecto final — Procesamiento de Lenguaje Natural · Maestría en Matemáticas Aplicadas**
Miguel Castelblanco

Este notebook solo *narra* resultados: toda la lógica vive en `src/rag_mb` (fuente única de verdad).
Requisitos: `pip install -e .[dev]`, datos en `data/raw/` (ver `data/raw/README.md`) y `rmb build <saga>`.

In [ ]:
SAGA = "lotr"          # "dune" | "narnia" cuando estén los datos
import json, pandas as pd
from IPython.display import Image, display, Markdown
from rag_mb.config import paths
from rag_mb.ingest.build import build_saga, units_for
P = paths(); FIG = P.figures / SAGA

## 1. Ingesta: unidades citables
Libro → capítulos (regex + limpieza de cabeceras). Guion → escenas, clasificando cada línea por su **coordenada x** en el PDF (formato estándar de la industria).

In [ ]:
# build_saga(SAGA)   # descomentar la primera vez (≈1 min)
U = units_for(SAGA)
df = pd.DataFrame([{"source": u.source_id, "work": u.work, "medium": u.medium, "words": u.n_words, "lines": len(u.lines)} for u in U])
df.groupby(["source", "work"]).agg(units=("words", "size"), words=("words", "sum"), lines=("lines", "sum"))

## 2. Estadística léxica
Zipf (MCO y MLE), Heaps, divergencia de Jensen-Shannon y log-odds con prior de Dirichlet (Monroe et al. 2008).

In [ ]:
from rag_mb.analysis.pipeline import analyze_saga
S = analyze_saga(SAGA)          # ≈10 s con LSA;  method="neural" para sentence-transformers
pd.Series(S["lexical"]).round(3)

In [ ]:
display(Image(FIG / "zipf.png"))
display(pd.read_csv(FIG / "logodds_film_over_book.csv").head(15))
display(pd.read_csv(FIG / "logodds_book_over_film.csv").head(15))

## 3. Mapa de adaptación: transporte óptimo
$a$ = masa de cada segmento del libro (∝ palabras), $b$ = masa de cada escena, $C_{ij} = 1-\cos(x_i, y_j)$ en un espacio LSA común.

* **Balanceado** (Sinkhorn): $\min_{P\in U(a,b)} \langle P,C\rangle - \varepsilon H(P)$ → proyección baricéntrica y fidelidad al orden (Spearman).
* **No balanceado** (KL, $\tau$): el guion no tiene que cubrir todo el libro; la retención $\rho_i=(P\mathbf 1)_i/a_i$ detecta capítulos omitidos.

In [ ]:
for w, v in S["works"].items():
    print(f"{w}: Spearman orden={v['order_spearman_ot']:.2f}  costo OT={v['ot_cost_balanced']:.3f}")
    print("   menos retenidos:", [c.split('—')[-1].strip() for c, _ in v["least_retained"][:3]])
    display(Image(FIG / f"ot_plan_{w}.png"))

In [ ]:
f = FIG / "saga_chapter_to_film.csv"
if f.exists():   # solo sagas con más de una película
    display(Image(FIG / "ot_plan_saga.png"))
    display(pd.read_csv(f).query("status != 'en su lugar'"))
else:
    print("La saga tiene una sola película: no aplica el mapa capítulo → película.")


## 4. Desplazamiento semántico (PPMI-SVD + Procrustes ortogonal)

In [ ]:
pd.read_csv(FIG / "semantic_shift.csv").head(20)

## 5. Redes de personajes (Laplaciano normalizado, Fiedler, centralidad)

In [ ]:
g = S["graphs"]
print(f"τ de Kendall (centralidad libro vs película) = {g['kendall_tau']:.2f} (p={g['kendall_p']:.1e})")
print(f"λ2 libro = {g['fiedler_book']:.3f}   λ2 película = {g['fiedler_film']:.3f}")
display(pd.DataFrame({"libro": g["fiedler_partition_book"], "película": g["fiedler_partition_film"]}))
pd.read_csv(FIG / "character_centrality.csv")

## 6. RAG: métricas de recuperación (sin LLM)

In [ ]:
from rag_mb.eval.runner import retrieval_eval
R = retrieval_eval(SAGA, k=8)
display(R.groupby("mode")[["recall@k", "hit@k", "mrr", "ndcg@k"]].mean().round(3))
R.pivot_table(index="category", columns="mode", values="recall@k").round(2)

## 7. Benchmark: LLM general vs RAG
```
rmb eval lotr --client openai --conditions closed_book rag_bm25_routed rag_hybrid_routed
rmb eval lotr --client gemini --conditions closed_book rag_bm25_routed rag_hybrid_routed
rmb eval lotr --client manual   # sin API: exporta prompts → pegar en ChatGPT/Gemini → --import
```

In [ ]:
runs = sorted(r for r in (P.reports / "runs").glob(f"{SAGA}_*") if (r / "report.md").exists())
for r in runs[-3:]:
    display(Markdown((r / "report.md").read_text(encoding="utf-8")))